In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import joblib

In [2]:
data = {
    "Age": [21, 22, 20, 23, 21, 24, 22, 20, 23, 21,
            22, 24, 20, 21, 23, 22, 24, 20, 21, 23,
            22, 21, 24, 20, 23, 22, 21, 24, 20, 22],

    "CGPA": [3.2, 3.5, 2.8, 3.7, 3.1, 3.8, 3.4, 2.9, 3.6, 3.3,
             3.5, 3.9, 2.7, 3.2, 3.8, 3.6, 3.7, 2.8, 3.4, 3.9,
             3.3, 3.1, 3.8, 2.9, 3.7, 3.5, 3.2, 3.9, 2.6, 3.6],

    "City": ["Lahore", "Multan", "Lahore", "Karachi", "Multan",
             "Lahore", "Karachi", "Multan", "Lahore", "Karachi",
             "Multan", "Lahore", "Karachi", "Multan", "Lahore",
             "Karachi", "Multan", "Lahore", "Karachi", "Multan",
             "Lahore", "Karachi", "Multan", "Lahore", "Karachi",
             "Multan", "Lahore", "Karachi", "Multan", "Lahore"],

    "Education": ["BS", "BS", "MS", "BS", "MS", "MS", "BS", "BS",
                  "MS", "BS", "BS", "MS", "BS", "MS", "MS", "BS",
                  "MS", "BS", "BS", "MS", "BS", "MS", "MS", "BS",
                  "MS", "BS", "MS", "MS", "BS", "BS"],

    "Placed": [1, 1, 0, 1, 1, 1, 1, 0, 1, 1,
               1, 1, 0, 1, 1, 1, 1, 0, 1, 1,
               1, 1, 1, 0, 1, 1, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

print(df.head())

   Age  CGPA     City Education  Placed
0   21   3.2   Lahore        BS       1
1   22   3.5   Multan        BS       1
2   20   2.8   Lahore        MS       0
3   23   3.7  Karachi        BS       1
4   21   3.1   Multan        MS       1


In [3]:
print("Dataset Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (30, 5)

Data Types:
Age            int64
CGPA         float64
City             str
Education        str
Placed         int64
dtype: object


In [4]:
print(df["Placed"].value_counts())

print("\nClass Percentage:")
print(df["Placed"].value_counts(normalize=True) * 100)

Placed
1    24
0     6
Name: count, dtype: int64

Class Percentage:
Placed
1    80.0
0    20.0
Name: proportion, dtype: float64


In [5]:
X = df.drop("Placed", axis=1)
y = df["Placed"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (30, 4)
Target Shape: (30,)


In [6]:
numeric_features = ["Age", "CGPA"]

categorical_features = ["City", "Education"]

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['Age', 'CGPA']
Categorical: ['City', 'Education']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nTraining Data Types:")
print(X_train.dtypes)

Training Shape: (24, 4)
Testing Shape: (6, 4)
Training Target: (24,)
Testing Target: (6,)

Training Data Types:
Age            int64
CGPA         float64
City             str
Education        str
dtype: object


In [8]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [9]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(class_weight="balanced"))
])

In [10]:
pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


In [11]:
X_train_processed = pipeline.named_steps["preprocessing"].transform(X_train)

X_test_processed = pipeline.named_steps["preprocessing"].transform(X_test)

print("Original Train Shape:", X_train.shape)
print("Processed Train Shape:", X_train_processed.shape)
print("Processed Train Data Type:", X_train_processed.dtype)

print("\nOriginal Test Shape:", X_test.shape)
print("Processed Test Shape:", X_test_processed.shape)
print("Processed Test Data Type:", X_test_processed.dtype)

Original Train Shape: (24, 4)
Processed Train Shape: (24, 7)
Processed Train Data Type: float64

Original Test Shape: (6, 4)
Processed Test Shape: (6, 7)
Processed Test Data Type: float64


In [12]:
predictions = pipeline.predict(X_test)

print("Predictions:", predictions)
print("Actual Values:", y_test.values)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Predictions: [0 1 1 1 1 1]
Actual Values: [0 1 1 1 1 1]
Accuracy: 1.0


In [13]:
joblib.dump(pipeline, "student_placement_pipeline.pkl")

print("Pipeline saved successfully!")

Pipeline saved successfully!


In [14]:
loaded_pipeline = joblib.load("student_placement_pipeline.pkl")

print("Pipeline loaded successfully!")

Pipeline loaded successfully!


In [15]:
new_students = pd.DataFrame({
    "Age": [22, 25, 21],
    "CGPA": [3.8, 2.7, 3.4],
    "City": ["Lahore", "Islamabad", "Multan"],
    "Education": ["BS", "MS", "BS"]
})

print(new_students)

print("\nShape:", new_students.shape)
print("\nData Types:")
print(new_students.dtypes)

   Age  CGPA       City Education
0   22   3.8     Lahore        BS
1   25   2.7  Islamabad        MS
2   21   3.4     Multan        BS

Shape: (3, 4)

Data Types:
Age            int64
CGPA         float64
City             str
Education        str
dtype: object


In [16]:
new_predictions = loaded_pipeline.predict(new_students)

print("New Predictions:", new_predictions)

for i, prediction in enumerate(new_predictions):

    if prediction == 1:
        print("Student", i + 1, ": Placed")
    else:
        print("Student", i + 1, ": Not Placed")

New Predictions: [1 1 1]
Student 1 : Placed
Student 2 : Placed
Student 3 : Placed
